# 21.18. MP-06 — суррогат движения MeshGraphNet

**Статус: инвентаризация и технический блокер; обучения и прогноза сети нет.** Проверяемый вопрос: может ли суррогат, обученный без данных проверочного человека, предсказать его движение лучше простого неподвижного контроля? В этом ноутбуке не представлены численные ошибки MeshGraphNet, потому что подходящая модель не запускалась.


## Основной план проверки на Нике

Ник выбран проверочным человеком: все его 12 фаз и поверхности должны быть исключены из обучения, построения PCA, настройки параметров и отбора эпох. При выводе разрешена лишь заранее выбранная КТ/маска фазы 00; остальные фазы скрыты до оценки. [Неподвижный контроль Ника](21.19_MP07_Прогноз_по_одной_КТ.ipynb) даёт средний Dice `0,9177` к автоматическим маскам остальных фаз. Подходящих проверенных весов и обучающих согласованных сеток пока нет, поэтому ни один прогноз MeshGraphNet на Нике не получен.


## Доступность модели и входов (обновлено 30.09.2026)

MONAI Physio `2026.09.1` установлен локально. В текущем загрузчике документированы готовые веса движения лёгкого; исторические **сердечные** веса найдены отдельно в официальном выпуске `2026.07.3`. Checkpoint указывает 28 внешних обучающих случаев `pm*`, 15 коэффициентов PCA и шаблон из 167 240 вершин. Ника в перечне обучения нет. Существующие STL Ника не имеют подтверждённого соответствия вершинам шаблона, а автоматическая `whole_heart` маска может обозначать другую анатомическую область.

Готовые веса, шаблон и общие рёбра доступны локально в исключённом `runs/`. Крупный исторический `pca_model.json` пока недоступен полностью: Git LFS не завершил получение, прямой сервер отдавал файл с непригодной скоростью и оборвал передачу. Медицинские данные и веса не добавлены в Git.

## Условие будущего честного теста

Разбиение выполняется **по человеку до** построения PCA, согласованной поверхности, настройки гиперпараметров и обучения. Все фазы тестового человека остаются вне этих шагов. На этапе прогноза допустима только его заранее выбранная опорная 3D КТ/маска и заранее разрешённые параметры; остальные реально снятые фазы открываются после фиксации прогноза. Для любых предложенных весов требуется манифест обучавшихся людей и исследований, анатомических классов, версий, контрольной суммы и правил подготовки сеток. Без такого манифеста отсутствие утечки не установлено.

Критерий будущего сравнения — пофазные геометрические ошибки на исключённом человеке и выигрыш относительно неподвижной маски [MP-07](21.19_MP07_Прогноз_по_одной_КТ.ipynb). Отдельно нужны QC складок/якобиана, топологии перенесённой сетки, выходов за FOV и, после геометрического QC, FEM-ответа при одинаковой электрической постановке. Процент R–R не приравнивается к механической фазе.


## Что встроено и что удалось запустить

[Текущий список загрузок](https://project-monai.github.io/monai-physio/cli_scripts/download_data.html) содержит готовый checkpoint **лёгкого**, но не сердца. [История официальных выпусков](https://github.com/Project-MONAI/monai-physio/releases) подтверждает, что сердечные веса входили в Git LFS версии `2026.07.3` и были удалены из репозитория в `2026.08.0`. Значит, код прогноза встроен, а готовая сердечная модель существует как исторический артефакт, который нужно брать отдельно.

[Технический тест](mp06_legacy_checkpoint_smoke.json) запустил **неизменённые веса** в API `2026.09.1` после сопоставления старых имён полей нормировки и имени файла шаблона. На среднем шаблоне сеть выдала конечные смещения для стадий 0 и 0,5; средние нормы `0,146` и `0,392` мм. Это доказывает исполнимость checkpoint, но не качество прогноза для Ника. Обучение и адаптация на наших данных не выполнялись.

## Решение

Готовая сердечная сеть технически работает. Честный персональный тест пока не завершён: нужна **готовая** PCA-модель, подгонка её к одной исходной КТ Ника с QC соответствия анатомии и заранее зафиксированный прогноз до открытия остальных фаз. Никаких выводов об улучшении относительно неподвижного контроля, влиянии на FEM или пригодности для расчёта функций сердца пока нет. Обучение на данных проекта не входит в постановку. Подробности — в [журнале MP-05/06/07](mp05_07_results.md).